In [38]:
"""
Amplicon-level frameshift/revertant analysis with niraparib response plots.
Frameshift sequences can be exchanged for clone of interest.

This script merges CRISPResso allele frequency txt files, translates each allele,
classifies alleles relative to a user-defined frameshift clone and wild-type
amplicon, collapses technical replicates, calculates log2 fold-change relative
to Day 7, and generates:

    1. Allele-class log2FC bar plot with replicate dots and p-value brackets.
    2. Revertant/frameshift protein-alignment heatmap with frequency and log2FC.

Input files are expected to be tab-delimited .txt files containing at least:
    Aligned_Sequence, Reference_Sequence, #Reads

Sample file names should contain condition and replicate labels such as:
    day_7_a, mock_a, 1_nira_a

Edit only the settings block below for different amplicons, frameshift clones,
or niraparib concentrations.
"""

from __future__ import annotations

import itertools
import re
from functools import reduce
from pathlib import Path

import matplotlib as mpl
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from Bio.Seq import Seq
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------

BASE_DIR = Path("/Users/annahoracek/Desktop/Hockemeyer/NGS/2025/10_2_25/exon_3_lenti_genomic")
INPUT_DIR = BASE_DIR / "1A_input_x3"
OUTPUT_DIR = BASE_DIR / "1A_output_test"

# Amplicon translation settings.
START_CODON = "TTAGGA"
STOP_CODON = "AAAGAG"
EXPECTED_TERMINAL_RESIDUE = "E"

# Protein references for this amplicon/clone.
WT_PROTEIN = "LGPISLNWFEELSSEAPPYNSEPAEESEHKNNNYEPNLFKTPQRKPSYNQLASTPIIFKE"
FRAMESHIFT_PROTEINS = {
    "I79fs": [
        "LGPISLNWFEELSSEAPPYNSEPAEESEHKNNNYEPNLFKTPQRKPSYNQLASTPIYSK",
        "LGPISLNWFEELSSEAPPYNSEPAEESEHKNNNYEPNLFKTPQRKPSYNQLASTPIYSK*",
    ]
}
PRIMARY_FRAMESHIFT_LABEL = "I80Yfs"
PRIMARY_FRAMESHIFT_SUMMARY = "p.I80Yfs*"

# Allele class labels.
REVERTANT_LABEL = "Reversion"
COMPOUND_FRAMESHIFT_LABEL = "Compound frameshift"
WILD_TYPE_LABEL = "Wild-type"
OTHER_LABEL = "Other"

# Experimental conditions.
BASELINE_CONDITION = {"prefix": "day_7", "label": "Day 7", "color": "gray"}
MOCK_CONDITION = {"prefix": "mock", "label": "Mock", "color": "deepskyblue"}

# Add, remove, or reorder niraparib concentrations here.
NIRAPARIB_CONCENTRATIONS = [
    {"prefix": "1_nira", "label": "1uM Nira", "color": "goldenrod"},
]

# Only these replicates are used. Control/no-Cas9 replicates are intentionally ignored.
REPLICATES = ["a", "b", "c"]

# Bar plot settings.
BAR_PLOT_CONDITIONS = [MOCK_CONDITION, *NIRAPARIB_CONCENTRATIONS]
MUTATION_ORDER = [PRIMARY_FRAMESHIFT_LABEL, COMPOUND_FRAMESHIFT_LABEL, REVERTANT_LABEL]
BAR_PLOT_TITLE = "Exon 3 - amplicon 1A"
BAR_PLOT_FILENAME = "allele_log2fc.pdf"

# Revertant heatmap settings.
HEATMAP_MIN_NIRA_MEAN = 0.05
HEATMAP_TYPES_REGEX = f"{PRIMARY_FRAMESHIFT_LABEL}|{REVERTANT_LABEL}"
HEATMAP_FILENAME = "all_revertants_niraparib_heatmap.pdf"
LOG2FC_VMIN = -3
LOG2FC_VMAX = 3
ND_COLOR = "#D9D9D9"

#provides extra feedback
VERBOSE = True


# -----------------------------------------------------------------------------
# Matplotlib defaults
# -----------------------------------------------------------------------------

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Arial"]


# -----------------------------------------------------------------------------
# Sequence utilities
# -----------------------------------------------------------------------------

def reverse_complement(seq: str) -> str:
    """Return the reverse complement of a DNA sequence."""
    complement = str.maketrans("ACGTNacgtn", "TGCANtgcan")
    return str(seq).translate(complement)[::-1]


def translate_sequence(sequence: str, start_codon: str, stop_codon: str) -> str:
    """Translate the sequence between the first start and stop codon matches."""
    sequence = str(sequence).upper().replace("-", "")
    start_codon = start_codon.upper()
    stop_codon = stop_codon.upper()

    start_idx = sequence.find(start_codon)
    stop_idx = sequence.find(stop_codon)

    if start_idx == -1 or stop_idx == -1 or stop_idx < start_idx:
        return ""

    coding_sequence = sequence[start_idx : stop_idx + len(stop_codon)]
    return str(Seq(coding_sequence).translate(table="Standard", to_stop=True))


def add_translation_column(
    df: pd.DataFrame,
    start_codon: str = START_CODON,
    stop_codon: str = STOP_CODON,
    sequence_col: str = "aligned_sequence",
) -> pd.DataFrame:
    """Add a Translation column to an allele table."""
    df = df.copy()
    df["Translation"] = df[sequence_col].apply(
        lambda sequence: translate_sequence(sequence, start_codon, stop_codon)
    )
    return df


def mark_putative_frameshifts(
    df: pd.DataFrame,
    translation_col: str = "Translation",
    terminal_residue: str = EXPECTED_TERMINAL_RESIDUE,
) -> pd.DataFrame:
    """Append '*' to products that do not end with the expected terminal residue."""
    df = df.copy()
    translations = df[translation_col].fillna("").astype(str)
    mask = ~translations.str.endswith("*") & ~translations.str.endswith(terminal_residue)
    df.loc[mask, translation_col] = translations.loc[mask].str.rstrip() + "*"
    return df


def align_by_shared_ends(wt_seq: str, mut_seq: str) -> tuple[str, str]:
    """Align two short protein sequences by preserving common prefix and suffix."""
    start = 0
    while start < min(len(wt_seq), len(mut_seq)) and wt_seq[start] == mut_seq[start]:
        start += 1

    end = 0
    while (
        end < len(wt_seq) - start
        and end < len(mut_seq) - start
        and wt_seq[-(end + 1)] == mut_seq[-(end + 1)]
    ):
        end += 1

    wt_mid = wt_seq[start : len(wt_seq) - end]
    mut_mid = mut_seq[start : len(mut_seq) - end]

    if len(wt_mid) > len(mut_mid):
        mut_mid += "-" * (len(wt_mid) - len(mut_mid))
    elif len(mut_mid) > len(wt_mid):
        wt_mid += "-" * (len(mut_mid) - len(wt_mid))

    aligned_wt = wt_seq[:start] + wt_mid + wt_seq[len(wt_seq) - end :]
    aligned_mut = mut_seq[:start] + mut_mid + mut_seq[len(mut_seq) - end :]

    aligned_wt = aligned_wt[: len(wt_seq)].ljust(len(wt_seq), "-")
    aligned_mut = aligned_mut[: len(wt_seq)].ljust(len(wt_seq), "-")

    return aligned_wt, aligned_mut


def count_mutations_and_deletions(translation: str, template: str = WT_PROTEIN) -> tuple[int, int]:
    """Count amino-acid substitutions and deletions relative to a template."""
    translation = translation or ""
    seq_len = len(translation)
    template_len = len(template)

    if seq_len == template_len:
        deletions = translation.count("-")
        mutations = sum(
            aa != ref and aa != "-"
            for aa, ref in zip(translation, template)
        )
    elif seq_len < template_len:
        deletions = template_len - seq_len
        mutations = sum(
            aa != ref
            for aa, ref in zip(translation, template[:seq_len])
        )
    else:
        deletions = 0
        mutations = sum(
            aa != ref and aa != "-"
            for aa, ref in zip(translation[:template_len], template)
        )

    return int(mutations), int(deletions)


# -----------------------------------------------------------------------------
# Input and replicate handling
# -----------------------------------------------------------------------------

def read_count_file(file_path: Path) -> pd.DataFrame | None:
    """Read a CRISPResso-style count file and add sample-specific read columns."""
    try:
        df = pd.read_csv(file_path, sep="\t")
    except Exception as exc:
        print(f"Skipping {file_path.name}: could not read file ({exc})")
        return None

    if "Aligned_Sequence" not in df.columns:
        print(f"Skipping {file_path.name}: missing Aligned_Sequence column")
        return None

    sample_name = file_path.stem
    rename_dict = {}

    if "#Reads" in df.columns:
        rename_dict["#Reads"] = f"{sample_name}_#Reads"
    if "%Reads" in df.columns:
        rename_dict["%Reads"] = f"{sample_name}_%Reads"

    df = df.rename(columns=rename_dict)
    df["Reverse_Complement"] = df["Aligned_Sequence"].apply(reverse_complement)

    keep_cols = [
        "Aligned_Sequence",
        "Reference_Sequence",
        "Reverse_Complement",
        "n_inserted",
        "n_deleted",
        "n_mutated",
    ] + list(rename_dict.values())
    keep_cols = [col for col in keep_cols if col in df.columns]

    return df[keep_cols]


def merge_count_files(input_dir: Path) -> pd.DataFrame:
    """Merge all input .txt count files without writing intermediate files."""
    dfs = [read_count_file(path) for path in sorted(input_dir.glob("*.txt"))]
    dfs = [df for df in dfs if df is not None]

    if not dfs:
        raise FileNotFoundError(f"No valid .txt count files found in {input_dir}")

    merge_cols = [
        "Aligned_Sequence",
        "Reference_Sequence",
        "Reverse_Complement",
        "n_inserted",
        "n_deleted",
        "n_mutated",
    ]
    merge_cols = [col for col in merge_cols if all(col in df.columns for df in dfs)]

    merged = reduce(
        lambda left, right: pd.merge(left, right, on=merge_cols, how="outer"),
        dfs,
    ).fillna(0)

    read_cols = [col for col in merged.columns if col.endswith("_#Reads")]
    merged["common_count"] = merged[read_cols].gt(0).sum(axis=1)
    merged["total_reads"] = merged[read_cols].sum(axis=1)

    merged = merged.sort_values(
        by=["common_count", "total_reads"],
        ascending=False,
    )

    merged = merged.drop(
        columns=["common_count", "total_reads", "Reverse_Complement"],
        errors="ignore",
    )
    merged.columns = merged.columns.str.lower()

    percent_cols = [col for col in merged.columns if "%" in col]
    return merged.drop(columns=percent_cols, errors="ignore")


def condition_prefixes() -> list[str]:
    """Return all condition prefixes used to collapse read counts."""
    prefixes = [BASELINE_CONDITION["prefix"], MOCK_CONDITION["prefix"]]
    prefixes += [dose["prefix"] for dose in NIRAPARIB_CONCENTRATIONS]
    return prefixes


def combine_technical_replicates(
    df: pd.DataFrame,
    prefixes: list[str] | None = None,
    replicates: list[str] = REPLICATES,
) -> pd.DataFrame:
    """Sum technical read columns and convert each biological replicate to percent reads.

    Only replicate labels listed in REPLICATES are used. Any no-Cas9/control
    columns remain ignored unless their labels are explicitly added to REPLICATES.
    """
    df = df.copy()
    prefixes = prefixes or condition_prefixes()

    drop_cols = [
        col for col in df.columns
        if "%reads" in col or col == "total num reads" or col == "reference_sequence" or "n_" in col
    ]
    df = df.drop(columns=drop_cols, errors="ignore")

    for prefix, replicate in itertools.product(prefixes, replicates):
        sample_key = f"{prefix}_{replicate}"
        matching_cols = [col for col in df.columns if re.search(sample_key, col)]

        if not matching_cols:
            continue

        sum_col = f"{sample_key}_read_sum"
        pct_col = f"{sample_key}%reads"
        df[sum_col] = df[matching_cols].sum(axis=1)

        total_reads = df[sum_col].sum()
        df[pct_col] = (df[sum_col] / total_reads) * 100 if total_reads else 0

    read_cols = [col for col in df.columns if "#reads" in col or col.endswith("_read_sum")]
    return df.drop(columns=read_cols, errors="ignore")


# -----------------------------------------------------------------------------
# Classification and summaries
# -----------------------------------------------------------------------------

def classify_translation(translation: str) -> str:
    """Classify one translated allele using user-defined amplicon settings."""
    if not translation:
        return OTHER_LABEL

    if translation == WT_PROTEIN:
        return WILD_TYPE_LABEL

    for label, sequences in FRAMESHIFT_PROTEINS.items():
        if translation in sequences:
            return label

    if translation.endswith(EXPECTED_TERMINAL_RESIDUE):
        return REVERTANT_LABEL

    return COMPOUND_FRAMESHIFT_LABEL


def add_allele_annotations(df: pd.DataFrame) -> pd.DataFrame:
    """Add type, mutation/deletion counts, and summary labels."""
    df = df.copy()
    df = mark_putative_frameshifts(df)
    df["type"] = df["Translation"].fillna("").astype(str).apply(classify_translation)

    counts = df["Translation"].fillna("").astype(str).apply(
        lambda seq: count_mutations_and_deletions(seq, WT_PROTEIN)
    )
    df["num_mutated"] = [item[0] for item in counts]
    df["num_deleted"] = [item[1] for item in counts]

    df["summary"] = (
        df["type"].astype(str)
        + "_"
        + df["num_mutated"].astype(str)
        + "_mutated_"
        + df["num_deleted"].astype(str)
        + "_deleted"
    )

    df.loc[df["type"] == PRIMARY_FRAMESHIFT_LABEL, "summary"] = PRIMARY_FRAMESHIFT_SUMMARY
    return df


def summarize_by_type(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse alleles by type, summing percent reads."""
    read_cols = [col for col in df.columns if col.endswith("%reads")]
    return df.groupby("type", as_index=False)[read_cols].sum()


def summarize_by_translation(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse alleles by translation, summing percent reads."""
    read_cols = [col for col in df.columns if col.endswith("%reads")]
    collapsed = df.groupby("Translation", as_index=False)[read_cols].sum()
    collapsed = add_allele_annotations(collapsed)
    return collapsed


def condition_columns() -> dict[str, list[str]]:
    """Build a condition-to-percent-read-column mapping for biological replicates only."""
    mapping = {
        BASELINE_CONDITION["label"]: [
            f"{BASELINE_CONDITION['prefix']}_{rep}%reads" for rep in REPLICATES
        ],
        MOCK_CONDITION["label"]: [
            f"{MOCK_CONDITION['prefix']}_{rep}%reads" for rep in REPLICATES
        ],
    }

    for dose in NIRAPARIB_CONCENTRATIONS:
        mapping[dose["label"]] = [f"{dose['prefix']}_{rep}%reads" for rep in REPLICATES]

    return mapping


def add_condition_stats(
    df: pd.DataFrame,
    conditions: dict[str, list[str]],
    keep_cols: list[str] | None = None,
) -> pd.DataFrame:
    """Keep requested identifiers, raw percent columns, and condition mean/std."""
    keep_cols = keep_cols or []
    keep_cols = [col for col in keep_cols if col in df.columns]
    stats = df[keep_cols].copy() if keep_cols else df.copy()

    for condition, columns in conditions.items():
        existing = [col for col in columns if col in df.columns]
        if not existing:
            if VERBOSE:
                print(f"No columns available for {condition}")
            stats[f"{condition}_mean"] = np.nan
            stats[f"{condition}_std"] = np.nan
            continue

        for col in existing:
            if col not in stats.columns:
                stats[col] = df[col]

        stats[f"{condition}_mean"] = df[existing].mean(axis=1)
        stats[f"{condition}_std"] = df[existing].std(axis=1)

    return stats


def add_log2_fold_changes(
    df: pd.DataFrame,
    comparison_conditions: list[dict[str, str]] | None = None,
    reference_prefix: str | None = None,
    replicates: list[str] = REPLICATES,
) -> pd.DataFrame:
    """Calculate replicate-level and mean log2FC relative to Day 7.

    No pseudocount is used. Infinite values from division by zero are converted
    to NaN before calculating the replicate mean.
    """
    df = df.copy()
    comparison_conditions = comparison_conditions or [MOCK_CONDITION, *NIRAPARIB_CONCENTRATIONS]
    reference_prefix = reference_prefix or BASELINE_CONDITION["prefix"]

    for condition in comparison_conditions:
        prefix = condition["prefix"]
        for replicate in replicates:
            numerator = f"{prefix}_{replicate}%reads"
            denominator = f"{reference_prefix}_{replicate}%reads"
            output = f"{prefix}_{replicate}_log2FC"

            if numerator not in df.columns or denominator not in df.columns:
                df[output] = np.nan
                continue

            with np.errstate(divide="ignore", invalid="ignore"):
                df[output] = np.log2(df[numerator] / df[denominator])

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    for condition in comparison_conditions:
        prefix = condition["prefix"]
        replicate_cols = [f"{prefix}_{replicate}_log2FC" for replicate in replicates]
        df[f"{prefix}_mean_log2FC"] = df[replicate_cols].mean(axis=1, skipna=True)
        df[f"{condition['label']}_log2FC"] = df[f"{prefix}_mean_log2FC"]

    return df


def build_type_plot_dataframe(type_summary: pd.DataFrame) -> pd.DataFrame:
    """Add condition stats/log2FC and keep columns needed for bar plotting."""
    stats = add_condition_stats(
        type_summary,
        condition_columns(),
        keep_cols=["type"],
    )
    with_log2fc = add_log2_fold_changes(stats)

    keep = ["type"]
    keep += [col for col in with_log2fc.columns if "log2fc" in col.lower()]
    return with_log2fc[keep]


def build_translation_heatmap_dataframe(translation_summary: pd.DataFrame) -> pd.DataFrame:
    """Add stats/log2FC for translation-level heatmaps."""
    stats = add_condition_stats(
        translation_summary,
        condition_columns(),
        keep_cols=["Translation", "type", "summary"],
    )
    return add_log2_fold_changes(stats)


# -----------------------------------------------------------------------------
# Bar plot
# -----------------------------------------------------------------------------

def reps_for_type(df: pd.DataFrame, allele_type: str, condition: dict[str, str]) -> np.ndarray:
    """Return finite replicate log2FC values for one allele type/condition."""
    cols = [f"{condition['prefix']}_{rep}_log2FC" for rep in REPLICATES]
    cols = [col for col in cols if col in df.columns]
    if not cols:
        return np.array([])

    values = df.loc[df["type"] == allele_type, cols].to_numpy().flatten()
    return values[np.isfinite(values)]


def bar_height(df: pd.DataFrame, allele_type: str, condition: dict[str, str]) -> float:
    """Return the mean log2FC value for one allele type/condition."""
    col = f"{condition['prefix']}_mean_log2FC"
    values = df.loc[df["type"] == allele_type, col].to_numpy().flatten()
    return float(values[0]) if len(values) else np.nan


def welch_pvalue(a: np.ndarray, b: np.ndarray, n_perm: int = 20_000) -> float:
    """Return Welch's t-test p-value, with a permutation fallback if SciPy is absent."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if a.size < 2 or b.size < 2:
        return np.nan

    try:
        from scipy.stats import ttest_ind

        return float(ttest_ind(a, b, equal_var=False, nan_policy="omit").pvalue)
    except Exception:
        rng = np.random.default_rng(0)
        observed = abs(a.mean() - b.mean())
        pooled = np.concatenate([a, b])
        n_a = a.size
        count = 0

        for _ in range(n_perm):
            rng.shuffle(pooled)
            difference = abs(pooled[:n_a].mean() - pooled[n_a:].mean())
            count += difference >= observed

        return (count + 1) / (n_perm + 1)


def pvalue_to_stars(pvalue: float) -> str:
    """Convert p-value to a significance label."""
    if not np.isfinite(pvalue):
        return "n/a"
    if pvalue < 0.001:
        return "***"
    if pvalue < 0.01:
        return "**"
    if pvalue < 0.05:
        return "*"
    return "ns"


def draw_significance_bar(ax: plt.Axes, x1: float, x2: float, y: float, h: float, pvalue: float) -> None:
    """Draw a significance bracket."""
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c="black")
    ax.text(
        (x1 + x2) / 2,
        y + h * 1.05,
        pvalue_to_stars(pvalue),
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )


def plot_allele_type_log2fc(df_plot: pd.DataFrame, output_path: Path) -> None:
    """Plot allele-class log2FC bars with replicate dots and p-value brackets."""
    df = df_plot.copy()
    df["type"] = df["type"].astype(str).str.strip()
    df = df[df["type"].isin(MUTATION_ORDER)]
    df["type"] = pd.Categorical(df["type"], categories=MUTATION_ORDER, ordered=True)
    df = df.sort_values("type")

    x = np.arange(len(BAR_PLOT_CONDITIONS))
    bar_width = 0.8 / len(MUTATION_ORDER)
    offsets = np.linspace(-0.4 + bar_width / 2, 0.4 - bar_width / 2, len(MUTATION_ORDER))

    fig, ax = plt.subplots(figsize=(5.5, 5))
    colors = ["deepskyblue", "limegreen", "goldenrod", "deeppink", "darkorange"]

    for i, allele_type in enumerate(MUTATION_ORDER):
        xi = x + offsets[i]
        heights = [bar_height(df, allele_type, condition) for condition in BAR_PLOT_CONDITIONS]

        ax.bar(
            xi,
            heights,
            width=bar_width,
            label=allele_type,
            color=colors[i % len(colors)],
            edgecolor="black",
        )

        for j, condition in enumerate(BAR_PLOT_CONDITIONS):
            reps = reps_for_type(df, allele_type, condition)
            if not len(reps):
                continue

            jitter = (np.random.rand(len(reps)) - 0.5) * bar_width * 0.7
            ax.scatter(
                xi[j] + jitter,
                reps,
                s=30,
                edgecolors="black",
                color=colors[i % len(colors)],
                zorder=3,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [condition["label"] for condition in BAR_PLOT_CONDITIONS],
        fontsize=14,
        fontweight="bold",
        rotation=45,
    )
    ax.set_ylabel(r"log$_2$FC[treatment/day 7]", fontsize=14, fontweight="bold")
    ax.set_title(BAR_PLOT_TITLE, fontsize=18, fontweight="bold", pad=12)
    ax.tick_params(axis="y", labelsize=14)

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin if ymax > ymin else 1
    tick_h = 0.01 * yrange
    stack_step = 0.05 * yrange
    top_pad = 0.018 * yrange

    baseline_type = PRIMARY_FRAMESHIFT_LABEL if PRIMARY_FRAMESHIFT_LABEL in df["type"].astype(str).values else None

    for j, condition in enumerate(BAR_PLOT_CONDITIONS):
        if baseline_type is None:
            continue

        base_reps = reps_for_type(df, baseline_type, condition)
        group_values = []

        for allele_type in MUTATION_ORDER:
            group_values.append(bar_height(df, allele_type, condition))
            group_values.extend(reps_for_type(df, allele_type, condition))

        group_values = np.array(group_values, dtype=float)
        group_values = group_values[np.isfinite(group_values)]
        if not group_values.size:
            continue

        y_base = group_values.max() + top_pad
        comparisons = [allele_type for allele_type in MUTATION_ORDER if allele_type != baseline_type]

        for k, allele_type in enumerate(comparisons):
            mutant_reps = reps_for_type(df, allele_type, condition)
            pvalue = welch_pvalue(base_reps, mutant_reps)
            x1 = (x + offsets[MUTATION_ORDER.index(baseline_type)])[j]
            x2 = (x + offsets[MUTATION_ORDER.index(allele_type)])[j]
            draw_significance_bar(ax, x1, x2, y_base + k * stack_step, tick_h, pvalue)

    ax.legend(
        title="",
        fontsize=12,
        bbox_to_anchor=(1.02, 1),
        frameon=False,
        prop={"weight": "bold", "size": 12},
    )

    plt.tight_layout()
    fig.savefig(output_path, format="pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)


# -----------------------------------------------------------------------------
# Heatmap plot
# -----------------------------------------------------------------------------

def compute_annot_fontsize(n_rows: int, n_cols: int, base: int = 12) -> int:
    """Scale heatmap annotation font size by matrix shape."""
    size = base - 0.12 * n_rows - 0.6 * max(0, n_cols - 2)
    return int(np.clip(size, 6, 12))


def ordered_heatmap_conditions() -> list[dict[str, str]]:
    """Return conditions shown in log2FC heatmap."""
    return [MOCK_CONDITION, *NIRAPARIB_CONCENTRATIONS]


def frequency_conditions() -> list[dict[str, str]]:
    """Return conditions shown in frequency heatmap."""
    return [BASELINE_CONDITION, MOCK_CONDITION, *NIRAPARIB_CONCENTRATIONS]


def plot_revertant_heatmap(df: pd.DataFrame, output_path: Path) -> None:
    """Plot aligned revertant/frameshift sequences with log2FC and frequency heatmaps."""
    df = df.copy()

    nira_mean_cols = [f"{dose['label']}_mean" for dose in NIRAPARIB_CONCENTRATIONS]
    existing_nira_means = [col for col in nira_mean_cols if col in df.columns]

    type_mask = df["summary"].str.contains(HEATMAP_TYPES_REGEX, case=False, na=False)
    abundance_mask = pd.Series(False, index=df.index)

    if existing_nira_means:
        abundance_mask = df[existing_nira_means].max(axis=1) > HEATMAP_MIN_NIRA_MEAN

    primary_mask = df["summary"].str.contains(
        PRIMARY_FRAMESHIFT_SUMMARY,
        case=False,
        na=False,
    )

    df = df[type_mask & (abundance_mask | primary_mask)].copy()

    if df.empty:
        raise ValueError("No rows available for heatmap after filtering.")

    df["translation_length"] = df["Translation"].astype(str).str.len()
    df["is_primary"] = df["summary"].str.contains(
        PRIMARY_FRAMESHIFT_SUMMARY,
        case=False,
        na=False,
    )
    df["is_revertant"] = df["summary"].str.contains(
        REVERTANT_LABEL,
        case=False,
        na=False,
    )

    df = df.sort_values(
        by=["is_primary", "is_revertant", "translation_length"],
        ascending=[True, False, False],
    ).drop(columns=["is_primary", "is_revertant", "translation_length"])

    df_no_wt = df[df["Translation"] != WT_PROTEIN].copy()

    if df_no_wt.empty:
        raise ValueError("No non-wild-type rows available for heatmap.")

    # ------------------------------------------------------------------
    # Build log2FC and frequency matrices
    # ------------------------------------------------------------------
    logfc_cols = [f"{condition['label']}_log2FC" for condition in ordered_heatmap_conditions()]
    logfc_cols = [col for col in logfc_cols if col in df_no_wt.columns]

    heat_df = df_no_wt.set_index("Translation")[logfc_cols]
    heat_df.columns = [col.replace("_log2FC", "") for col in heat_df.columns]
    heat_df.replace([np.inf, -np.inf], np.nan, inplace=True)

    freq_cols = [f"{condition['label']}_mean" for condition in frequency_conditions()]
    freq_cols = [col for col in freq_cols if col in df_no_wt.columns]

    freq_df = df_no_wt.set_index("Translation")[freq_cols]
    freq_df = freq_df.reindex(heat_df.index)

    # ------------------------------------------------------------------
    # Build sequence alignment display
    # ------------------------------------------------------------------
    seqs_no_wt = df_no_wt["Translation"].tolist()
    all_seqs = seqs_no_wt + [WT_PROTEIN]
    max_len = max(len(seq) for seq in all_seqs)

    aligned_seqs = [align_by_shared_ends(WT_PROTEIN, seq)[1] for seq in seqs_no_wt]
    aligned_wt_seq = align_by_shared_ends(WT_PROTEIN, WT_PROTEIN)[0]

    aa_colors = {
        **dict.fromkeys(list("DECMKRSTFYNQGLVIAWHP"), "#CCCCCC"),
        "-": "#FFFFFF",
        "*": "#FF0000",
    }

    height = max(6, len(seqs_no_wt) * 0.3 + 1)
    width = 8 + max_len * 0.2

    fig = plt.figure(figsize=(width, height))
    gs = gridspec.GridSpec(
        2,
        3,
        width_ratios=[max_len, max(4, max(1, heat_df.shape[1]) * 3), 8],
        height_ratios=[len(seqs_no_wt), 1],
        hspace=0.02,
        wspace=0.3,
    )

    ax_align = fig.add_subplot(gs[0, 0])
    ax_heat = fig.add_subplot(gs[0, 1])
    ax_freq = fig.add_subplot(gs[0, 2])
    ax_wt = fig.add_subplot(gs[1, 0])
    ax_blank = fig.add_subplot(gs[1, 1])
    ax_blank2 = fig.add_subplot(gs[1, 2])

    ax_blank.axis("off")
    ax_blank2.axis("off")

    # ------------------------------------------------------------------
    # Draw aligned mutant sequences
    # ------------------------------------------------------------------
    ax_align.set_xlim(-1.5, max_len)
    ax_align.set_ylim(len(seqs_no_wt) - 0.5, -0.5)
    ax_align.axis("off")

    for row_i, seq in enumerate(aligned_seqs):
        for col_i, aa in enumerate(seq):
            if aa == "*":
                color = "#FF0000"
            elif aa == "-":
                color = "#FFFFFF"
            elif aa != aligned_wt_seq[col_i]:
                color = "#FFABBB"
            else:
                color = aa_colors.get(aa, "#CCCCCC")

            rect = plt.Rectangle(
                (col_i - 0.5, row_i - 0.5),
                1,
                1,
                facecolor=color,
                edgecolor="black",
                linewidth=0.3,
            )
            ax_align.add_patch(rect)
            ax_align.text(
                col_i,
                row_i,
                aa,
                ha="center",
                va="center",
                fontsize=11,
                color="black",
            )

    for row_i, summary in enumerate(df_no_wt["summary"]):
        ax_align.text(
            -1.5,
            row_i,
            summary,
            ha="right",
            va="center",
            fontsize=11,
        )

    # ------------------------------------------------------------------
    # Log2FC heatmap
    # Missing values are grey and labeled N.D. only once
    # ------------------------------------------------------------------
    log2fc_fontsize = 13
    heatmap_label_fontsize = 12
    nd_fontsize = 13

    data = heat_df.to_numpy(dtype=float)
    nan_mask = ~np.isfinite(data)

    plot_data = data.copy()
    plot_data[nan_mask] = 0

    annot_labels = np.empty_like(plot_data, dtype=object)
    annot_labels[:] = ""
    annot_labels[~nan_mask] = np.vectorize(lambda x: f"{x:.2f}")(data[~nan_mask])

    cmap = sns.color_palette("vlag", as_cmap=True)
    plot_df = pd.DataFrame(plot_data, index=heat_df.index, columns=heat_df.columns)

    sns_h = sns.heatmap(
        plot_df,
        ax=ax_heat,
        cmap=cmap,
        center=0,
        vmin=LOG2FC_VMIN,
        vmax=LOG2FC_VMAX,
        annot=annot_labels,
        fmt="",
        annot_kws={
            "fontsize": log2fc_fontsize,
            "color": "black",
            "fontweight": "bold",
        },
        linewidths=0.5,
        linecolor="gray",
        cbar=False,
    )

    # Overlay grey boxes and N.D. text only for missing cells
    for row_i, col_i in zip(*np.where(nan_mask)):
        rect = plt.Rectangle(
            (col_i, row_i),
            1,
            1,
            facecolor=ND_COLOR,
            edgecolor="gray",
            linewidth=0.5,
            zorder=2,
        )
        ax_heat.add_patch(rect)
        ax_heat.text(
            col_i + 0.5,
            row_i + 0.5,
            "N.D.",
            ha="center",
            va="center",
            fontsize=nd_fontsize,
            fontweight="bold",
            color="black",
            zorder=3,
        )

    for spine in ax_heat.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_edgecolor("black")

    ax_heat.set_xticks(np.arange(len(heat_df.columns)) + 0.5)
    ax_heat.set_xticklabels(
        heat_df.columns,
        rotation=45,
        fontsize=heatmap_label_fontsize,
        fontweight="bold",
    )
    ax_heat.set_yticks([])
    ax_heat.set_xlabel("")
    ax_heat.set_ylabel("")

    cax_h = inset_axes(ax_heat, width="10%", height="30%", loc="right", borderpad=-1.4)
    cbar = plt.colorbar(sns_h.collections[0], cax=cax_h)
    cbar.set_label(
        "Log2FC",
        rotation=90,
        fontsize=13,
        labelpad=8,
        fontweight="bold",
    )
    cbar.ax.tick_params(labelsize=12, pad=2)

    for tick in cbar.ax.get_yticklabels():
        tick.set_fontweight("bold")

    # ------------------------------------------------------------------
    # Frequency heatmap
    # ------------------------------------------------------------------
    vmax_freq = np.nanmax(freq_df.values.astype(float)) if freq_df.size else 1.0
    is_percent = np.isfinite(vmax_freq) and vmax_freq <= 1.0

    annot_vals = (freq_df * 100).round(1) if is_percent else freq_df.round(3)
    cbar_label = "Frequency (%)" if is_percent else "Frequency"

    sns_f = sns.heatmap(
        freq_df,
        ax=ax_freq,
        cmap=sns.color_palette("vlag", as_cmap=True),
        vmin=0,
        vmax=max(vmax_freq, 1),
        annot=annot_vals,
        fmt=".2f",
        annot_kws={
            "fontsize": 13,
            "color": "black",
            "fontweight": "bold",
        },
        linewidths=0.5,
        linecolor="gray",
        cbar=False,
    )

    for spine in ax_freq.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1)
        spine.set_edgecolor("black")

    ax_freq.set_xticks(np.arange(len(freq_df.columns)) + 0.5)
    ax_freq.set_xticklabels(
        freq_df.columns,
        rotation=45,
        fontsize=heatmap_label_fontsize,
        fontweight="bold",
    )
    ax_freq.set_yticks([])
    ax_freq.set_xlabel("")
    ax_freq.set_ylabel("")

    cax_f = inset_axes(ax_freq, width="10%", height="30%", loc="right", borderpad=-1.4)
    cbar_f = plt.colorbar(sns_f.collections[0], cax=cax_f)
    cbar_f.set_label(
        cbar_label,
        rotation=90,
        fontsize=13,
        labelpad=8,
        fontweight="bold",
    )
    cbar_f.ax.tick_params(labelsize=12, pad=2)

    for tick in cbar_f.ax.get_yticklabels():
        tick.set_fontweight("bold")

    # ------------------------------------------------------------------
    # Draw WT sequence row
    # ------------------------------------------------------------------
    ax_wt.set_xlim(-1.5, max_len)
    ax_wt.set_ylim(0.5, -0.5)
    ax_wt.axis("off")

    wt_seq_pad = WT_PROTEIN.ljust(max_len, "-").strip("-")

    for col_i, aa in enumerate(wt_seq_pad):
        rect = plt.Rectangle(
            (col_i - 0.5, -0.5),
            1,
            1,
            facecolor=aa_colors.get(aa, "#CCCCCC"),
            edgecolor="black",
            linewidth=0.3,
        )
        ax_wt.add_patch(rect)
        ax_wt.text(
            col_i,
            0,
            aa,
            ha="center",
            va="center",
            fontsize=11,
            color="black",
        )

    ax_wt.text(
        -1.5,
        0,
        WILD_TYPE_LABEL,
        ha="right",
        va="center",
        fontsize=11,
        fontweight="bold",
    )

    plt.tight_layout(pad=0.7)
    fig.savefig(output_path, format="pdf", dpi=300, bbox_inches="tight")
    plt.close(fig)

# -----------------------------------------------------------------------------
# Main analysis
# -----------------------------------------------------------------------------

def main() -> None:
    if not INPUT_DIR.exists():
        raise FileNotFoundError(f"Input directory does not exist: {INPUT_DIR}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    merged = merge_count_files(INPUT_DIR)
    merged = add_translation_column(merged)
    merged = add_allele_annotations(merged)
    merged.to_csv(OUTPUT_DIR / "merged_output.csv", index=False)

    allele_table = combine_technical_replicates(merged)
    allele_table = add_allele_annotations(allele_table)
    allele_table.to_csv(OUTPUT_DIR / "collapsed_alleles.csv", index=False)

    type_summary = summarize_by_type(allele_table)
    type_summary.to_csv(OUTPUT_DIR / "allele_type_summary.csv", index=False)

    type_plot = build_type_plot_dataframe(type_summary)
    type_plot.to_csv(OUTPUT_DIR / "allele_type_log2fc.csv", index=False)
    plot_allele_type_log2fc(type_plot, OUTPUT_DIR / BAR_PLOT_FILENAME)

    translation_summary = summarize_by_translation(allele_table)
    translation_summary.to_csv(OUTPUT_DIR / "translation_summary.csv", index=False)

    translation_heatmap = build_translation_heatmap_dataframe(translation_summary)
    translation_heatmap.to_csv(OUTPUT_DIR / "translation_log2fc_heatmap_input.csv", index=False)
    plot_revertant_heatmap(translation_heatmap, OUTPUT_DIR / HEATMAP_FILENAME)

    print(f"Analysis complete. Results saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

/Users/annahoracek/opt/anaconda3/lib/python3.9/site-packages/Bio/Seq.py:2804: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
/var/folders/md/qj_hgg097yq2z5t14lb2pm280000gn/T/ipykernel_65017/1965219962.py:1058: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0.7)


Analysis complete. Results saved to: /Users/annahoracek/Desktop/Hockemeyer/NGS/2025/10_2_25/exon_3_lenti_genomic/1A_output_test
